# Análise da Camada Silver — CVM Data For All

**Parte 1:** Análise exploratória do universo de empresas selecionadas  
**Parte 2:** Testes de sanidade das tabelas Silver (BP, DRE, DFC)

**Inputs:**
- `layer_01_bronze.cad_cia_aberta`
- `layer_02_silver.n0_empresas_selecionadas`
- `layer_02_silver.n1_bp` · `n1_dre` · `n1_dfc`

**Outputs:** Visualizações exploratórias + Relatório de sanidade (sem escrita no banco)

In [ ]:
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from dotenv import load_dotenv

# Carrega .env da raiz do projeto (funciona de qualquer subpasta)
for _env in ['.env', '../.env', '../../.env']:
    if os.path.exists(_env):
        load_dotenv(_env)
        break

def create_db_engine():
    user = quote_plus(os.getenv('DB_USER', 'postgres'))
    password = quote_plus(os.getenv('DB_PASS', 'password'))
    host = os.getenv('DB_HOST', 'localhost')
    port = os.getenv('DB_PORT', '5432')
    dbname = os.getenv('DB_NAME', 'cvm_data_for_all')
    return create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}")

engine = create_db_engine()
print("✅ Conexão com o banco de dados estabelecida!")

---
# Parte 1 — Análise Exploratória: Universo de Empresas Selecionadas

**Objetivo:** Entender quem são as ~214 empresas elegíveis para análise financeira — distribuição setorial, geográfica e estrutura de controle — antes de processar qualquer demonstrativo.  
**Input:** `layer_02_silver.n0_empresas_selecionadas`  
**Output:** Visualizações exploratórias (sem escrita no banco)

## 🧮 Por que excluímos empresas do setor financeiro?

A exclusão de bancos, seguradoras e financeiras **não é uma preferência arbitrária** — é uma exigência técnica de **comparabilidade contábil**. Cada tipo de empresa segue padrões contábeis completamente distintos:

| Tipo de Empresa | Padrão Contábil | Por que não é comparável? |
|:---|:---|:---|
| Indústrias, Varejo, Serviços | **IFRS / BR GAAP** | Padrão base da nossa análise |
| **Bancos** | **COSIF** (Banco Central) | Sem Ativo Circulante tradicional; receita = *Margem Financeira*, não *Receita Bruta* |
| **Seguradoras** | **SUSEP** | Demonstrativos usam *Prêmios Ganhos* e *Sinistros*; sem EBITDA comparável |
| **Financeiras** | Misto COSIF/IFRS | Carteira de crédito como ativo principal — estrutura de BP completamente distinta |
| **Securitizadoras** | IFRS especializado | Ativo baseado em carteiras de recebíveis (CRI, CRA, FIDC) |
| **Holdings (Adm. Part.)** | IFRS (consolidado) | Resultado dominado por equivalência patrimonial, não por operações reais |

> **Nossa escolha metodológica:** Focamos exclusivamente em empresas **não-financeiras** e **operacionais** listadas na **B3**, garantindo que todos os demonstrativos financeiros sigam a mesma estrutura e possam ser comparados de forma significativa.

---
## 1. Carregando o Universo Completo da CVM

Antes de aplicar qualquer filtro, carregamos o cadastro completo de companhias da CVM para entender o universo de partida.

In [ ]:
print("📥 Carregando o universo completo de empresas da CVM...")

with engine.connect() as conn:
    df_all = pd.read_sql(
        text("SELECT * FROM layer_01_bronze.cad_cia_aberta"),
        con=conn
    )

print(f"✅ Total de registros no cadastro CVM: {len(df_all):,}")
df_all[["CNPJ_CIA", "DENOM_SOCIAL", "SIT", "TP_MERC", "SIT_EMISSOR", "SETOR_ATIV", "UF", "CONTROLE_ACIONARIO"]].head()

---
## 2. O Funil de Filtragem: Como Chegamos ao Nosso Universo de Análise

Cada filtro da query é aplicado de forma incremental, permitindo visualizar exatamente quantas empresas são eliminadas em cada etapa — e o **motivo** de cada exclusão.

In [ ]:
print("🔍 Construindo o funil de filtragem passo a passo...")

passos = []

passos.append({"filtro": "0. Universo CVM Completo", "n": len(df_all), "excluidas": 0,
               "motivo": "Todos os registros no cadastro da CVM"})

df1 = df_all[df_all["SIT"] == "ATIVO"]
passos.append({"filtro": "1. Situação: ATIVO", "n": len(df1), "excluidas": len(df_all) - len(df1),
               "motivo": "Remove canceladas, extintas e suspensas"})

df2 = df1[df1["TP_MERC"] == "BOLSA"]
passos.append({"filtro": "2. Mercado: BOLSA", "n": len(df2), "excluidas": len(df1) - len(df2),
               "motivo": "Apenas companhias listadas na B3"})

df3 = df2[df2["SIT_EMISSOR"] == "FASE OPERACIONAL"]
passos.append({"filtro": "3. Emissor: FASE OPERACIONAL", "n": len(df3), "excluidas": len(df2) - len(df3),
               "motivo": "Remove pré-operacionais e em registro"})

df4 = df3[~df3["SETOR_ATIV"].str.contains("Emp. Adm. Part", na=False, regex=False)]
passos.append({"filtro": "4. Excl. Holdings (Adm. Part.)", "n": len(df4), "excluidas": len(df3) - len(df4),
               "motivo": "Resultado dominado por equivalência patrimonial"})

df5 = df4[~df4["SETOR_ATIV"].str.contains("Banc", na=False, regex=False)]
passos.append({"filtro": "5. Excl. Bancos", "n": len(df5), "excluidas": len(df4) - len(df5),
               "motivo": "Padrão COSIF — incompatível com análise industrial"})

df6 = df5[~df5["SETOR_ATIV"].str.contains("Segurad", na=False, regex=False)]
passos.append({"filtro": "6. Excl. Seguradoras", "n": len(df6), "excluidas": len(df5) - len(df6),
               "motivo": "Padrão SUSEP — sem EBITDA comparável"})

df7 = df6[~df6["SETOR_ATIV"].str.contains("Financeira", na=False, regex=False)]
passos.append({"filtro": "7. Excl. Financeiras", "n": len(df7), "excluidas": len(df6) - len(df7),
               "motivo": "Carteira de crédito como ativo principal"})

df8 = df7[~df7["SETOR_ATIV"].str.contains("Securitiz", na=False, regex=False)]
passos.append({"filtro": "8. Excl. Securitizadoras", "n": len(df8), "excluidas": len(df7) - len(df8),
               "motivo": "Estrutura de ativo baseada em recebíveis (CRI/CRA/FIDC)"})

df_funil = pd.DataFrame(passos)
print(f"\n🎯 Universo final: {len(df8):,} empresas comparáveis")
print(f"📉 Total excluído: {len(df_all) - len(df8):,} empresas ({(1 - len(df8)/len(df_all))*100:.1f}% do universo inicial)")
df_funil[["filtro", "n", "excluidas", "motivo"]]

In [ ]:
cores = ["#2d0057", "#46008a", "#5e00bc", "#7519cc",
         "#8b35d4", "#a055dc", "#b575e4", "#ca95eb", "#dfb8f2"]

fig = go.Figure(go.Funnel(
    y=df_funil["filtro"].tolist(),
    x=df_funil["n"].tolist(),
    textposition="inside",
    textinfo="value+percent initial",
    opacity=0.9,
    marker={"color": cores[:len(df_funil)], "line": {"width": 1, "color": "white"}},
    connector={"line": {"color": "#ccc", "width": 1, "dash": "dot"}}
))
fig.update_layout(
    title={"text": "🔍 Funil de Seleção: Do Universo CVM ao Universo de Análise Financeira",
           "font": {"size": 17}, "x": 0.5, "xanchor": "center"},
    height=640, font={"family": "Arial", "size": 13},
    plot_bgcolor="white", paper_bgcolor="white",
    margin={"l": 250, "r": 80, "t": 80, "b": 40}
)
fig.show()

In [ ]:
df_excl = df_funil[df_funil["excluidas"] > 0].copy()

fig = px.bar(
    df_excl, x="excluidas", y="filtro", orientation="h", text="excluidas",
    color="excluidas", color_continuous_scale="Purples", hover_data={"motivo": True},
    title="❌ Empresas Eliminadas em Cada Etapa do Filtro",
    labels={"excluidas": "Empresas Eliminadas", "filtro": "Filtro Aplicado", "motivo": "Motivo"}
)
fig.update_traces(texttemplate="%{text:,}", textposition="outside")
fig.update_layout(
    height=420, showlegend=False, coloraxis_showscale=False,
    font={"family": "Arial", "size": 12}, plot_bgcolor="white", paper_bgcolor="white",
    xaxis={"showgrid": True, "gridcolor": "#eee"},
    yaxis={"categoryorder": "total ascending"},
    margin={"l": 280, "r": 100, "t": 60, "b": 50}
)
fig.show()

---
## 3. Empresas Excluídas: Um Olhar sobre os Setores Financeiros

Quais são exatamente os setores que ficaram de fora? Vamos detalhar as categorias excluídas pelos filtros de setor — lembrando que a exclusão ocorre **após** os filtros de situação cadastral (ou seja, são empresas ativas, listadas e operacionais que foram descartadas por incompatibilidade contábil).

In [ ]:
mascara_financeiro = (
    df3["SETOR_ATIV"].str.contains("Emp. Adm. Part", na=False, regex=False) |
    df3["SETOR_ATIV"].str.contains("Banc", na=False, regex=False) |
    df3["SETOR_ATIV"].str.contains("Segurad", na=False, regex=False) |
    df3["SETOR_ATIV"].str.contains("Financeira", na=False, regex=False) |
    df3["SETOR_ATIV"].str.contains("Securitiz", na=False, regex=False)
)
df_excluidas_setor = df3[mascara_financeiro].copy()

print(f"📊 Total de empresas excluídas por setor: {len(df_excluidas_setor):,}")
print(f"   (de um total de {len(df3):,} empresas ativas, listadas e operacionais)")

df_setores_excl = (
    df_excluidas_setor.groupby("SETOR_ATIV").size()
    .reset_index(name="count").sort_values("count", ascending=False)
)

fig = px.bar(
    df_setores_excl.sort_values("count", ascending=True),
    x="count", y="SETOR_ATIV", orientation="h", text="count",
    color="count", color_continuous_scale="Purples",
    title="🚫 Setores Excluídos da Análise (Empresas Ativas e Listadas na B3)",
    labels={"count": "Nº de Empresas", "SETOR_ATIV": "Setor de Atividade"}
)
fig.update_traces(texttemplate="%{text}", textposition="outside")
fig.update_layout(
    height=560, showlegend=False, coloraxis_showscale=False,
    font={"family": "Arial", "size": 11}, plot_bgcolor="white", paper_bgcolor="white",
    xaxis={"showgrid": True, "gridcolor": "#eee"},
    margin={"l": 310, "r": 80, "t": 60, "b": 50}
)
fig.show()

---
## 4. O Universo Final: Análise das Empresas Selecionadas

Agora que entendemos quem ficou de fora, vamos explorar **quem entrou**.

In [ ]:
df_selecionadas = df8.copy()

print("=" * 65)
print("  📊 RESUMO DO UNIVERSO DE ANÁLISE FINANCEIRA")
print("=" * 65)
print(f"  Total de empresas selecionadas : {len(df_selecionadas):>6,}")
print(f"  Total de setores distintos     : {df_selecionadas['SETOR_ATIV'].nunique():>6,}")
print(f"  Estados representados (UF)     : {df_selecionadas['UF'].nunique():>6,}")
print(f"  Tipos de controle acionário    : {df_selecionadas['CONTROLE_ACIONARIO'].nunique():>6,}")
print(f"  Categorias de registro CVM     : {df_selecionadas['CATEG_REG'].nunique():>6,}")
print("=" * 65)

### 4.1 Distribuição por Setor de Atividade

O **Treemap** abaixo mostra a hierarquia dos setores, com o tamanho de cada bloco proporcional ao número de empresas.

In [ ]:
df_por_setor = (
    df_selecionadas.groupby("SETOR_ATIV").size()
    .reset_index(name="count").sort_values("count", ascending=False)
)

fig = px.treemap(
    df_por_setor, path=["SETOR_ATIV"], values="count",
    color="count", color_continuous_scale="Purples",
    title="🗂️ Distribuição das Empresas Selecionadas por Setor de Atividade",
    custom_data=["count"]
)
fig.update_traces(
    texttemplate="<b>%{label}</b><br>%{customdata[0]} empresas",
    hovertemplate="<b>%{label}</b><br>Empresas: %{customdata[0]}<extra></extra>",
    textfont_size=12
)
fig.update_layout(height=580, font={"family": "Arial"}, coloraxis_showscale=False,
                  margin={"l": 10, "r": 10, "t": 60, "b": 10})
fig.show()

In [ ]:
fig = px.bar(
    df_por_setor.sort_values("count", ascending=True),
    x="count", y="SETOR_ATIV", orientation="h", text="count",
    color="count", color_continuous_scale="Purples",
    title="📊 Setores por Número de Empresas Selecionadas",
    labels={"count": "Nº de Empresas", "SETOR_ATIV": "Setor de Atividade"}
)
fig.update_traces(texttemplate="%{text}", textposition="outside")
fig.update_layout(
    height=660, showlegend=False, coloraxis_showscale=False,
    font={"family": "Arial", "size": 11}, plot_bgcolor="white", paper_bgcolor="white",
    yaxis={"categoryorder": "total ascending"}, xaxis={"showgrid": True, "gridcolor": "#eee"},
    margin={"l": 300, "r": 80, "t": 60, "b": 50}
)
fig.show()

### 4.2 Distribuição Geográfica por Estado (UF)

Onde estão sediadas as empresas do nosso universo de análise? É esperada uma forte concentração em São Paulo (SP), principal polo econômico e financeiro do Brasil.

In [ ]:
df_por_uf = (
    df_selecionadas.groupby("UF").size()
    .reset_index(name="count").sort_values("count", ascending=True)
)
fig = px.bar(
    df_por_uf, x="count", y="UF", orientation="h", text="count",
    color="count", color_continuous_scale="Purples",
    title="🗺️ Distribuição Geográfica: Empresas por Estado (UF)",
    labels={"count": "Nº de Empresas", "UF": "Estado (UF)"}
)
fig.update_traces(texttemplate="%{text}", textposition="outside")
fig.update_layout(
    height=500, showlegend=False, coloraxis_showscale=False,
    font={"family": "Arial", "size": 12}, plot_bgcolor="white", paper_bgcolor="white",
    xaxis={"showgrid": True, "gridcolor": "#eee"}, margin={"l": 60, "r": 80, "t": 60, "b": 50}
)
fig.show()

### 4.3 Controle Acionário

A maioria das empresas do nosso universo é de controle **privado** ou há representação significativa de empresas **públicas** (estatais)?

In [ ]:
df_controle = (
    df_selecionadas.groupby("CONTROLE_ACIONARIO").size()
    .reset_index(name="count").sort_values("count", ascending=True)
)
total_ctrl = df_controle["count"].sum()
df_controle["pct"] = (df_controle["count"] / total_ctrl * 100).round(1)
df_controle["label"] = df_controle.apply(lambda r: f"{r['count']}  ({r['pct']}%)", axis=1)
max_count = df_controle["count"].max()

fig = px.bar(
    df_controle, x="count", y="CONTROLE_ACIONARIO", orientation="h", text="label",
    color="count", color_continuous_scale="Purples",
    title="👥 Distribuição por Tipo de Controle Acionário",
    labels={"count": "Nº de Empresas", "CONTROLE_ACIONARIO": "Controle Acionário"}
)
fig.update_traces(textposition="outside")
fig.update_layout(
    height=400, showlegend=False, coloraxis_showscale=False,
    font={"family": "Arial", "size": 13}, plot_bgcolor="white", paper_bgcolor="white",
    xaxis={"showgrid": True, "gridcolor": "#eee", "range": [0, max_count * 1.4]},
    yaxis={"categoryorder": "total ascending"}, title={"x": 0.5, "xanchor": "center"},
    margin={"l": 160, "r": 40, "t": 70, "b": 50}
)
fig.show()

### 4.4 Categoria de Registro na CVM

A CVM classifica as companhias abertas em duas categorias:
- **Categoria A:** Pode emitir qualquer tipo de valor mobiliário, incluindo ações de livre negociação em bolsa.
- **Categoria B:** Restrita à emissão de valores mobiliários sem ações de livre negociação.

Como nosso filtro inclui apenas empresas listadas na **Bolsa**, esperamos predominância da Categoria A.

In [ ]:
df_categ = (
    df_selecionadas.groupby("CATEG_REG").size()
    .reset_index(name="count").sort_values("count", ascending=False)
)
fig = px.bar(
    df_categ, x="CATEG_REG", y="count", text="count",
    color="CATEG_REG", color_discrete_sequence=["#5e00bc", "#ca95eb"],
    title="📋 Distribuição por Categoria de Registro na CVM",
    labels={"count": "Nº de Empresas", "CATEG_REG": "Categoria de Registro"}
)
fig.update_traces(texttemplate="%{text}", textposition="outside")
fig.update_layout(
    height=420, showlegend=False, font={"family": "Arial", "size": 13},
    plot_bgcolor="white", paper_bgcolor="white",
    yaxis={"showgrid": True, "gridcolor": "#eee"},
    title={"x": 0.5, "xanchor": "center"}, margin={"l": 60, "r": 60, "t": 70, "b": 60}
)
fig.show()

---
## 5. Conclusões da Análise Exploratória

In [ ]:
setor_top = df_por_setor.iloc[0]
uf_top = df_por_uf.sort_values("count", ascending=False).iloc[0]
controle_top = df_controle.iloc[-1]

print("=" * 70)
print("  🎯 CONCLUSÕES DA ANÁLISE EXPLORATÓRIA")
print("=" * 70)
print(f"\n  📌 Universo de Partida:")
print(f"     {len(df_all):,} empresas no cadastro completo da CVM")
print(f"\n  🎯 Universo de Chegada:")
print(f"     {len(df8):,} empresas comparáveis")
print(f"     ({len(df8)/len(df_all)*100:.1f}% do total — {(1 - len(df8)/len(df_all))*100:.1f}% excluídos)")
print(f"\n  📊 Composição do Universo Final:")
print(f"     • Setor mais representado : {setor_top['SETOR_ATIV']}")
print(f"                                 ({setor_top['count']} empresas)")
print(f"     • Estado com mais sedes   : {uf_top['UF']} ({uf_top['count']} empresas)")
print(f"     • Controle predominante   : {controle_top['CONTROLE_ACIONARIO']}")
print(f"                                 ({controle_top['count']} empresas, "
      f"{controle_top['count']/len(df8)*100:.0f}% do total)")
print(f"\n  ✅ Este universo é a base de todas as análises financeiras")
print(f"     do pipeline de dados da CVM (BP, DRE e DFC).")
print("=" * 70)

---
---
# Parte 2 — Testes de Sanidade da Camada Silver

**Objetivo:** Validar a integridade das três tabelas N1 (BP, DRE, DFC) com testes formais: equação fundamental, integridade vertical pai-filho, flags de qualidade e duplicatas.

> 💡 **Regra de Ouro:** Este notebook deve ser executado sempre após uma carga ou refatoração da camada Silver. Se todos os blocos passarem, o pipeline está pronto para alimentar o dashboard.

In [ ]:
TABELA_BP  = 'layer_02_silver.n1_bp'
TABELA_DRE = 'layer_02_silver.n1_dre'
TABELA_DFC = 'layer_02_silver.n1_dfc'

print(f"📋 BP  : {TABELA_BP}")
print(f"📋 DRE : {TABELA_DRE}")
print(f"📋 DFC : {TABELA_DFC}")

---
# 📈 Testes de Sanidade — BP (`n1_bp`)

| # | Teste | O que verifica |
|---|---|---|
| 1 | **Visão Geral** | Volume, cobertura de empresas e períodos, nulos |
| 2 | **Equação Fundamental** | Ativo = Passivo + PL para cada empresa/data |
| 3 | **STATUS_MATH** | Distribuição dos resultados de validação contábil |
| 4 | **Flags de Qualidade** | Proporção de linhas reconstruídas e normalizadas |
| 5 | **IS_LEAF** | Sanidade da árvore hierárquica de contas |
| 6 | **Duplicatas** | Registros duplicados por empresa + data + conta |

---
## 🔍 BLOCO 1 — Visão Geral: Volume e Cobertura

In [ ]:
query_visao_geral = f"""
SELECT
    COUNT(*)                                          AS total_linhas,
    COUNT(DISTINCT "CNPJ_CIA")                        AS total_empresas,
    COUNT(DISTINCT "DT_REFER")                        AS total_periodos,
    MIN("DT_REFER")                                   AS periodo_mais_antigo,
    MAX("DT_REFER")                                   AS periodo_mais_recente,
    COUNT(DISTINCT "CD_CONTA")                        AS total_contas_distintas,
    SUM(CASE WHEN "VL_CONTA_TRATADO" IS NULL THEN 1 ELSE 0 END) AS nulls_vl_conta
FROM {TABELA_BP};
"""

with engine.connect() as conn:
    df_visao = pd.read_sql(text(query_visao_geral), conn)

print("📊 RESULTADO — Visão Geral BP")
display(df_visao)

nulls = df_visao['nulls_vl_conta'].iloc[0]
total = df_visao['total_linhas'].iloc[0]

if total == 0:
    print("❌ FALHOU: A tabela está VAZIA!")
elif nulls > 0:
    print(f"⚠️  ATENÇÃO: {nulls} valores NULL em VL_CONTA_TRATADO encontrados.")
else:
    print(f"✅ PASSOU: {total:,} linhas carregadas, sem NULLs em VL_CONTA_TRATADO.")

---
## ⚖️ BLOCO 2 — Equação Fundamental: Ativo = Passivo + PL

$$\text{Ativo} \, (\text{conta raiz } 1) = \text{Passivo + PL} \, (\text{conta raiz } 2)$$

Para cada combinação `CNPJ_CIA + DT_REFER`, a diferença entre Ativo e Passivo deve ser **zero** (tolerância de R$ 0,01 para arredondamentos).

In [ ]:
query_equacao = f"""
WITH totais AS (
    SELECT
        "CNPJ_CIA", "DENOM_CIA", "DT_REFER",
        SUM(CASE WHEN "CD_CONTA" = '1' THEN "VL_CONTA_TRATADO" ELSE 0 END) AS total_ativo,
        SUM(CASE WHEN "CD_CONTA" = '2' THEN "VL_CONTA_TRATADO" ELSE 0 END) AS total_passivo
    FROM {TABELA_BP}
    GROUP BY "CNPJ_CIA", "DENOM_CIA", "DT_REFER"
)
SELECT
    "CNPJ_CIA", "DENOM_CIA", "DT_REFER",
    ROUND(total_ativo::NUMERIC, 2)              AS total_ativo,
    ROUND(total_passivo::NUMERIC, 2)            AS total_passivo,
    ROUND((total_ativo - total_passivo)::NUMERIC, 2) AS diferenca,
    CASE
        WHEN ABS(total_ativo - total_passivo) <= 0.01 THEN '✅ OK'
        WHEN total_ativo = 0 AND total_passivo = 0    THEN '⚠️ ZERADO'
        ELSE '❌ DESBALANCEADO'
    END AS status_equacao
FROM totais
ORDER BY ABS(total_ativo - total_passivo) DESC;
"""

with engine.connect() as conn:
    df_equacao = pd.read_sql(text(query_equacao), conn)

n_desbalanceados = (df_equacao['status_equacao'] == '❌ DESBALANCEADO').sum()
n_zerados        = (df_equacao['status_equacao'] == '⚠️ ZERADO').sum()
n_ok             = (df_equacao['status_equacao'] == '✅ OK').sum()

print("📊 RESULTADO — Equação Fundamental por Empresa/Data")
print(f"\n✅ OK:             {n_ok}")
print(f"⚠️  ZERADO:         {n_zerados}")
print(f"❌ DESBALANCEADO:  {n_desbalanceados}")

if n_desbalanceados > 0:
    print(f"\n❌ FALHOU: {n_desbalanceados} combinações empresa/data com Ativo ≠ Passivo+PL!")
    display(df_equacao[df_equacao['status_equacao'] == '❌ DESBALANCEADO'].head(20))
else:
    print("\n✅ PASSOU: Equação fundamental validada em todos os períodos.")

---
## 📈 BLOCO 3 — STATUS_MATH: Distribuição dos Resultados de Validação

In [ ]:
query_status_math = f"""
SELECT
    "STATUS_MATH",
    COUNT(*)                                              AS qtd_linhas,
    COUNT(DISTINCT "CNPJ_CIA" || "DT_REFER"::TEXT)       AS qtd_empresa_data,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)   AS pct_total
FROM {TABELA_BP}
GROUP BY "STATUS_MATH"
ORDER BY qtd_linhas DESC;
"""

with engine.connect() as conn:
    df_status = pd.read_sql(text(query_status_math), conn)

print("📊 RESULTADO — Distribuição de STATUS_MATH (BP)")
display(df_status)

pct_desbalanceado = df_status.loc[df_status['STATUS_MATH'].isin(['DESBALANCEADO', 'DESEQUILIBRADO']), 'pct_total'].sum()
LIMITE_ACEITAVEL = 5.0

if pct_desbalanceado > LIMITE_ACEITAVEL:
    print(f"\n❌ FALHOU: {pct_desbalanceado:.1f}% das linhas estão DESBALANCEADAS (limite: {LIMITE_ACEITAVEL}%).")
else:
    print(f"\n✅ PASSOU: Apenas {pct_desbalanceado:.1f}% das linhas desbalanceadas (dentro do limite de {LIMITE_ACEITAVEL}%).")

---
## 🏷️ BLOCO 4 — Flags de Qualidade: Reconstrução e Normalização

In [ ]:
query_flags = f"""
SELECT
    "FLAG_RECONSTRUCAO", "FLAG_NORMALIZACAO",
    COUNT(*)                                            AS qtd_linhas,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_total
FROM {TABELA_BP}
GROUP BY "FLAG_RECONSTRUCAO", "FLAG_NORMALIZACAO"
ORDER BY qtd_linhas DESC;
"""

with engine.connect() as conn:
    df_flags = pd.read_sql(text(query_flags), conn)

print("📊 RESULTADO — Distribuição das Flags de Qualidade (BP)")
display(df_flags)

total_linhas = df_flags['qtd_linhas'].sum()
linhas_reconstruidas = df_flags.loc[df_flags['FLAG_RECONSTRUCAO'] == True, 'qtd_linhas'].sum()
pct_reconstruidas = (linhas_reconstruidas / total_linhas * 100) if total_linhas > 0 else 0

if pct_reconstruidas == 100:
    print("\n❌ ALERTA: 100% das linhas são reconstruídas — nenhum valor original da CVM foi preservado!")
elif pct_reconstruidas > 50:
    print(f"\n⚠️  ATENÇÃO: {pct_reconstruidas:.1f}% das linhas são reconstruídas. Verifique se está correto.")
else:
    print(f"\n✅ PASSOU: {pct_reconstruidas:.1f}% de linhas reconstruídas — proporção dentro do esperado.")

---
## 🌳 BLOCO 5 — IS_LEAF: Sanidade da Árvore Hierárquica

- `IS_LEAF = True` → conta analítica (folha). São as únicas que devem ser somadas no BI para evitar dupla contagem.
- `IS_LEAF = False` → conta sintética (nó pai).

In [ ]:
query_leaf = f"""
SELECT
    "IS_LEAF",
    COUNT(*)                                            AS qtd_linhas,
    COUNT(DISTINCT "CD_CONTA")                          AS contas_distintas,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_total
FROM {TABELA_BP}
GROUP BY "IS_LEAF"
ORDER BY "IS_LEAF";
"""

with engine.connect() as conn:
    df_leaf = pd.read_sql(text(query_leaf), conn)

print("📊 RESULTADO — Distribuição de IS_LEAF (BP)")
display(df_leaf)

valores_leaf = df_leaf['IS_LEAF'].astype(str).str.lower().tolist()
tem_nao_leaf = any(v in ['false', '0', 'no'] for v in valores_leaf)
tem_leaf     = any(v in ['true', '1', 'yes'] for v in valores_leaf)

if not tem_nao_leaf:
    print("\n❌ FALHOU: Todas as contas são IS_LEAF = True. Os nós pai (contas sintéticas) estão faltando!")
elif not tem_leaf:
    print("\n❌ FALHOU: Nenhuma conta IS_LEAF = True encontrada.")
else:
    print("\n✅ PASSOU: A árvore contém tanto contas analíticas (folhas) quanto sintéticas (pais).")

---
## 🔂 BLOCO 6 — Duplicatas: Extração Blindada

Para cada `CNPJ_CIA + DT_REFER + CD_CONTA + DS_CONTA` deve existir **no máximo uma linha**. Este teste deve retornar **zero linhas**.

In [ ]:
query_duplicatas = f"""
SELECT "CNPJ_CIA", "DENOM_CIA", "DT_REFER", "CD_CONTA", "DS_CONTA", COUNT(*) AS ocorrencias
FROM {TABELA_BP}
GROUP BY "CNPJ_CIA", "DENOM_CIA", "DT_REFER", "CD_CONTA", "DS_CONTA"
HAVING COUNT(*) > 1
ORDER BY ocorrencias DESC
LIMIT 20;
"""

with engine.connect() as conn:
    df_dupl = pd.read_sql(text(query_duplicatas), conn)

print("📊 RESULTADO — Duplicatas BP (esperado: zero linhas)")
if df_dupl.empty:
    print("✅ PASSOU: Nenhuma duplicata encontrada. Extração blindada funcionou corretamente.")
else:
    display(df_dupl)
    print(f"\n❌ FALHOU: {len(df_dupl)} grupos de duplicatas encontrados (mostrando até 20).")

In [ ]:
print("=" * 60)
print("  RELATÓRIO DE SANIDADE — layer_02_silver.n1_bp")
print("=" * 60)

resultados = [
    ("1. Visão Geral (volume e nulos)", (total > 0) and (nulls == 0)),
    ("2. Equação Fundamental (Ativo = Passivo+PL)", n_desbalanceados == 0),
    (f"3. STATUS_MATH (≤{LIMITE_ACEITAVEL}% desbalanceado)", pct_desbalanceado <= LIMITE_ACEITAVEL),
    ("4. Flags de Qualidade (não 100% reconstruído)", pct_reconstruidas < 100),
    ("5. IS_LEAF (árvore com pais e folhas)", tem_leaf and tem_nao_leaf),
    ("6. Duplicatas (zero registros)", df_dupl.empty),
]

aprovados = 0
for nome, passou in resultados:
    icone = "✅ PASSOU" if passou else "❌ FALHOU"
    print(f"  {icone}  |  {nome}")
    if passou:
        aprovados += 1

print("=" * 60)
print(f"  RESULTADO FINAL: {aprovados}/{len(resultados)} testes aprovados")
if aprovados == len(resultados):
    print("  🎉 BP APROVADO — dados prontos para o dashboard!")
else:
    print("  🚨 BP REPROVADO — revisar o pipeline Silver antes de usar no dashboard.")
print("=" * 60)

---
---
# 📈 Testes de Sanidade — DRE (`n1_dre`)

| # | Teste | O que verifica |
|---|---|---|
| 1 | **Visão Geral** | Volume, cobertura, nulos |
| 2 | **Cascata da DRE** | Integridade vertical pai-filho |
| 3 | **STATUS_MATH** | Distribuição de validação |
| 4 | **Flags de Qualidade** | Reconstruídas e normalizadas |
| 5 | **IS_LEAF** | Sanidade hierárquica |
| 6 | **Duplicatas** | Extração blindada |
| 7 | **Cross-check DRE ↔ BP** | Paridade de empresas e períodos |

In [ ]:
query_visao_geral_dre = f"""
SELECT
    COUNT(*)                                          AS total_linhas,
    COUNT(DISTINCT "CNPJ_CIA")                        AS total_empresas,
    COUNT(DISTINCT "DT_REFER")                        AS total_periodos,
    MIN("DT_REFER")                                   AS periodo_mais_antigo,
    MAX("DT_REFER")                                   AS periodo_mais_recente,
    COUNT(DISTINCT "CD_CONTA")                        AS total_contas_distintas,
    SUM(CASE WHEN "VL_CONTA_TRATADO" IS NULL THEN 1 ELSE 0 END) AS nulls_vl_conta
FROM {TABELA_DRE};
"""

with engine.connect() as conn:
    df_visao_dre = pd.read_sql(text(query_visao_geral_dre), conn)

print("📊 RESULTADO — Visão Geral DRE")
display(df_visao_dre)

nulls_dre = df_visao_dre['nulls_vl_conta'].iloc[0]
total_dre = df_visao_dre['total_linhas'].iloc[0]

if total_dre == 0:
    print("\n❌ FALHOU: A tabela DRE está VAZIA!")
elif nulls_dre > 0:
    print(f"\n⚠️  ATENÇÃO: {nulls_dre} valores NULL em VL_CONTA_TRATADO.")
else:
    print(f"\n✅ PASSOU: {total_dre:,} linhas carregadas, sem NULLs em VL_CONTA_TRATADO.")

---
## ⛲ BLOCO 2 — Cascata da DRE: Integridade Vertical Pai-Filho

O teste correto da DRE é checar **cada par pai-filho individualmente**, não somar todos os filhos de nível 2 (isso causaria dupla contagem por ser uma cascata).

In [ ]:
query_resumo_dre = f"""
WITH pares AS (
    SELECT
        pai."CNPJ_CIA", pai."DT_REFER",
        pai."VL_CONTA_TRATADO"                     AS vl_pai,
        SUM(filho."VL_CONTA_TRATADO")              AS soma_filhos,
        pai."FLAG_RECONSTRUCAO"                    AS pai_reconstruido
    FROM {TABELA_DRE} pai
    JOIN {TABELA_DRE} filho
        ON  pai."CNPJ_CIA"               = filho."CNPJ_CIA"
        AND pai."DT_REFER"               = filho."DT_REFER"
        AND filho."CD_CONTA"             LIKE pai."CD_CONTA" || '.' || '%'
        AND filho."CD_CONTA_QTD_DIGITOS" = pai."CD_CONTA_QTD_DIGITOS" + 2
    GROUP BY pai."CNPJ_CIA", pai."DT_REFER",
             pai."CD_CONTA", pai."VL_CONTA_TRATADO", pai."FLAG_RECONSTRUCAO"
)
SELECT
    CASE
        WHEN ABS(vl_pai - soma_filhos) <= 1000  THEN 'OK'
        WHEN vl_pai = 0 AND soma_filhos = 0     THEN 'ZERADO'
        ELSE 'DESBALANCEADO'
    END AS status_par,
    pai_reconstruido,
    COUNT(*) AS qtd
FROM pares
GROUP BY 1, 2
ORDER BY 1, 2;
"""

with engine.connect() as conn:
    df_resumo_dre = pd.read_sql(text(query_resumo_dre), conn)

print("📊 Resumo — Validação Vertical DRE (par pai-filho)")
display(df_resumo_dre)

n_erro_dre = int(df_resumo_dre.query("status_par == 'DESBALANCEADO' and not pai_reconstruido")['qtd'].sum()) if not df_resumo_dre.empty else 0
n_ok_dre = int(df_resumo_dre.query("status_par == 'OK'")['qtd'].sum()) if not df_resumo_dre.empty else 0

if n_erro_dre > 0:
    print(f"\n❌ FALHOU: {n_erro_dre} pares pai-filho DESBALANCEADOS (dados originais)")
else:
    print(f"\n✅ PASSOU: {n_ok_dre} pares pai-filho verificados — todos balanceados.")

ok_vertical_dre = (n_erro_dre == 0)

In [ ]:
query_status_math_dre = f"""
SELECT
    "STATUS_MATH",
    COUNT(*)                                              AS qtd_linhas,
    COUNT(DISTINCT "CNPJ_CIA" || "DT_REFER"::TEXT)       AS qtd_empresa_data,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)   AS pct_total
FROM {TABELA_DRE}
GROUP BY "STATUS_MATH"
ORDER BY qtd_linhas DESC;
"""
with engine.connect() as conn:
    df_status_dre = pd.read_sql(text(query_status_math_dre), conn)

print("📊 RESULTADO — Distribuição de STATUS_MATH (DRE)")
display(df_status_dre)

pct_desbalanceado_dre = df_status_dre.loc[
    df_status_dre['STATUS_MATH'].isin(['DESBALANCEADO', 'INCONSISTENTE']), 'pct_total'
].sum()
LIMITE = 5.0

if pct_desbalanceado_dre > LIMITE:
    print(f"\n❌ FALHOU: {pct_desbalanceado_dre:.1f}% das linhas DRE estão DESBALANCEADAS (limite: {LIMITE}%).")
else:
    print(f"\n✅ PASSOU: Apenas {pct_desbalanceado_dre:.1f}% desbalanceadas (dentro do limite de {LIMITE}%).")

In [ ]:
query_flags_dre = f"""
SELECT
    "FLAG_RECONSTRUCAO", "FLAG_NORMALIZACAO",
    COUNT(*)                                            AS qtd_linhas,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_total
FROM {TABELA_DRE}
GROUP BY "FLAG_RECONSTRUCAO", "FLAG_NORMALIZACAO"
ORDER BY qtd_linhas DESC;
"""
with engine.connect() as conn:
    df_flags_dre = pd.read_sql(text(query_flags_dre), conn)

print("📊 RESULTADO — Flags de Qualidade (DRE)")
display(df_flags_dre)

total_dre_f = df_flags_dre['qtd_linhas'].sum()
linhas_reconstruidas_dre = df_flags_dre.loc[df_flags_dre['FLAG_RECONSTRUCAO'] == True, 'qtd_linhas'].sum()
pct_reconstruidas_dre = (linhas_reconstruidas_dre / total_dre_f * 100) if total_dre_f > 0 else 0

if pct_reconstruidas_dre == 100:
    print("\n❌ ALERTA: 100% das linhas DRE são reconstruídas!")
elif pct_reconstruidas_dre > 50:
    print(f"\n⚠️  ATENÇÃO: {pct_reconstruidas_dre:.1f}% das linhas DRE são reconstruídas.")
else:
    print(f"\n✅ PASSOU: {pct_reconstruidas_dre:.1f}% de linhas reconstruídas.")

In [ ]:
query_leaf_dre = f"""
SELECT
    "IS_LEAF",
    COUNT(*)                                            AS qtd_linhas,
    COUNT(DISTINCT "CD_CONTA")                          AS contas_distintas,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_total
FROM {TABELA_DRE}
GROUP BY "IS_LEAF"
ORDER BY "IS_LEAF";
"""
with engine.connect() as conn:
    df_leaf_dre = pd.read_sql(text(query_leaf_dre), conn)

print("📊 RESULTADO — IS_LEAF (DRE)")
display(df_leaf_dre)

valores_leaf_dre = df_leaf_dre['IS_LEAF'].astype(str).str.lower().tolist()
tem_nao_leaf_dre = any(v in ['false', '0'] for v in valores_leaf_dre)
tem_leaf_dre     = any(v in ['true', '1'] for v in valores_leaf_dre)

if not tem_nao_leaf_dre:
    print("\n❌ FALHOU: Todas as contas DRE são IS_LEAF = True. Nós pai estão faltando!")
elif not tem_leaf_dre:
    print("\n❌ FALHOU: Nenhuma conta IS_LEAF = True na DRE.")
else:
    print("\n✅ PASSOU: Árvore DRE com contas analíticas (folhas) e sintéticas (pais).")

In [ ]:
query_duplicatas_dre = f"""
SELECT "CNPJ_CIA", "DENOM_CIA", "DT_REFER", "CD_CONTA", "DS_CONTA", COUNT(*) AS ocorrencias
FROM {TABELA_DRE}
GROUP BY "CNPJ_CIA", "DENOM_CIA", "DT_REFER", "CD_CONTA", "DS_CONTA"
HAVING COUNT(*) > 1
ORDER BY ocorrencias DESC LIMIT 20;
"""
with engine.connect() as conn:
    df_dupl_dre = pd.read_sql(text(query_duplicatas_dre), conn)

print("📊 RESULTADO — Duplicatas DRE (esperado: zero linhas)")
if df_dupl_dre.empty:
    print("✅ PASSOU: Nenhuma duplicata na DRE.")
else:
    display(df_dupl_dre)
    print(f"\n❌ FALHOU: {len(df_dupl_dre)} grupos de duplicatas na DRE.")

In [ ]:
query_crosscheck = f"""
WITH bp_cob AS (SELECT DISTINCT "CNPJ_CIA", "DT_REFER" FROM {TABELA_BP}),
dre_cob AS (SELECT DISTINCT "CNPJ_CIA", "DT_REFER" FROM {TABELA_DRE})
SELECT 'Só no BP (sem DRE)  ⚠️' AS situacao, COUNT(*) AS qtd_pares
FROM bp_cob b WHERE NOT EXISTS (SELECT 1 FROM dre_cob d WHERE d."CNPJ_CIA"=b."CNPJ_CIA" AND d."DT_REFER"=b."DT_REFER")
UNION ALL
SELECT 'Só na DRE (sem BP)  ⚠️', COUNT(*)
FROM dre_cob d WHERE NOT EXISTS (SELECT 1 FROM bp_cob b WHERE b."CNPJ_CIA"=d."CNPJ_CIA" AND b."DT_REFER"=d."DT_REFER")
UNION ALL
SELECT 'Presentes em ambos  ✅', COUNT(*)
FROM bp_cob b INNER JOIN dre_cob d ON b."CNPJ_CIA"=d."CNPJ_CIA" AND b."DT_REFER"=d."DT_REFER";
"""
with engine.connect() as conn:
    df_cross = pd.read_sql(text(query_crosscheck), conn)

print("📊 RESULTADO — Cross-check DRE ↔ BP")
display(df_cross)

so_bp  = df_cross.loc[df_cross['situacao'].str.contains('Só no BP'), 'qtd_pares'].sum()
so_dre = df_cross.loc[df_cross['situacao'].str.contains('Só na DRE'), 'qtd_pares'].sum()
ok7_dre = (so_bp == 0 and so_dre == 0)

if ok7_dre:
    print("\n✅ PASSOU: Cobertura idêntica entre BP e DRE.")
else:
    print(f"\n⚠️  ATENÇÃO: {so_bp} pares só no BP e {so_dre} pares só na DRE.")

In [ ]:
print("=" * 62)
print("  RELATÓRIO DE SANIDADE — layer_02_silver.n1_dre")
print("=" * 62)

resultados_dre = [
    ("1. Visão Geral (volume e nulos)", (total_dre > 0) and (nulls_dre == 0)),
    ("2. Cascata DRE (par pai-filho balanceado)", n_erro_dre == 0),
    ("3. STATUS_MATH (≤5% desbalanceado)", pct_desbalanceado_dre <= 5.0),
    ("4. Flags de Qualidade (não 100% reconstruído)", pct_reconstruidas_dre < 100),
    ("5. IS_LEAF (árvore com pais e folhas)", tem_leaf_dre and tem_nao_leaf_dre),
    ("6. Duplicatas (zero registros)", df_dupl_dre.empty),
    ("7. Cross-check DRE ↔ BP (sem lacunas)", ok7_dre),
]

aprovados_dre = 0
for nome, passou in resultados_dre:
    icone = "✅ PASSOU" if passou else "❌ FALHOU"
    print(f"  {icone}  |  {nome}")
    if passou:
        aprovados_dre += 1

print("=" * 62)
print(f"  RESULTADO FINAL: {aprovados_dre}/{len(resultados_dre)} testes aprovados")
if aprovados_dre == len(resultados_dre):
    print("  🎉 DRE APROVADA — dados prontos para análise financeira!")
else:
    print("  🚨 DRE REPROVADA — revisar o pipeline Silver antes de usar.")
print("=" * 62)

---
---
# 💵 Testes de Sanidade — DFC (`n1_dfc`)

A DFC parte do **lucro líquido da DRE** e explica como ele se transformou em **variação de caixa no BP**.

```
DRE → Lucro Líquido
         ↓
DFC → 6.01  Fluxo Operacional
      6.02  Fluxo de Investimento
      6.03  Fluxo de Financiamento
      6.04  Variação Cambial (opcional)
      ─────────────────────────────────
      6.05  Variação Líquida de Caixa
         ↓
BP  → 1.01.01 + 1.01.02  (Caixa + Aplicações de Liquidez)
```

In [ ]:
query_visao_geral_dfc = f"""
SELECT
    COUNT(*)                                          AS total_linhas,
    COUNT(DISTINCT "CNPJ_CIA")                        AS total_empresas,
    COUNT(DISTINCT "DT_REFER")                        AS total_periodos,
    MIN("DT_REFER")                                   AS periodo_mais_antigo,
    MAX("DT_REFER")                                   AS periodo_mais_recente,
    COUNT(DISTINCT "CD_CONTA")                        AS total_contas_distintas,
    SUM(CASE WHEN "VL_CONTA_TRATADO" IS NULL THEN 1 ELSE 0 END) AS nulls_vl_conta
FROM {TABELA_DFC};
"""
with engine.connect() as conn:
    df_visao_dfc = pd.read_sql(text(query_visao_geral_dfc), conn)

print("📊 RESULTADO — Visão Geral DFC")
display(df_visao_dfc)

nulls_dfc = df_visao_dfc['nulls_vl_conta'].iloc[0]
total_dfc = df_visao_dfc['total_linhas'].iloc[0]

if total_dfc == 0:
    print("\n❌ FALHOU: A tabela DFC está VAZIA!")
elif nulls_dfc > 0:
    print(f"\n⚠️  ATENÇÃO: {nulls_dfc} valores NULL em VL_CONTA_TRATADO.")
else:
    print(f"\n✅ PASSOU: {total_dfc:,} linhas carregadas, sem NULLs em VL_CONTA_TRATADO.")

---
## ⛲ BLOCO 2 — Equação Fundamental da DFC

$$6.01 + 6.02 + 6.03 + 6.04_{\text{(se houver)}} = 6.05$$

> 💡 `6.04` está presente apenas em empresas com operações em moeda estrangeira.

In [ ]:
query_identidade_dfc = f"""
WITH identidade AS (
    SELECT
        "CNPJ_CIA", "DENOM_CIA", "DT_REFER",
        SUM(CASE WHEN "CD_CONTA" IN ('6.01','6.02','6.03','6.04')
                 THEN "VL_CONTA_TRATADO" ELSE 0 END)  AS soma_fluxos,
        SUM(CASE WHEN "CD_CONTA" = '6.05'
                 THEN "VL_CONTA_TRATADO" ELSE 0 END)  AS variacao_caixa,
        BOOL_OR(CASE WHEN "CD_CONTA" IN ('6.01','6.02','6.03','6.04','6.05')
                     THEN "FLAG_RECONSTRUCAO" END)     AS reconstruido
    FROM {TABELA_DFC}
    WHERE "CD_CONTA" IN ('6.01','6.02','6.03','6.04','6.05')
    GROUP BY "CNPJ_CIA", "DENOM_CIA", "DT_REFER"
)
SELECT
    "CNPJ_CIA", "DENOM_CIA", "DT_REFER",
    ROUND(soma_fluxos::NUMERIC, 2)                    AS soma_fluxos,
    ROUND(variacao_caixa::NUMERIC, 2)                 AS variacao_caixa,
    ROUND((soma_fluxos - variacao_caixa)::NUMERIC, 2) AS diferenca,
    reconstruido,
    CASE
        WHEN soma_fluxos = 0 AND variacao_caixa = 0  THEN '⚠️ ZERADO'
        WHEN ABS(soma_fluxos - variacao_caixa) <= 1  THEN '✅ OK'
        WHEN reconstruido = FALSE
         AND ABS(soma_fluxos - variacao_caixa)
             / NULLIF(ABS(variacao_caixa), 0) * 100 <= 1.0
            THEN '💱 Efeito Cambial (6.04 implícito)'
        ELSE '❌ DIVERGE'
    END AS status_identidade
FROM identidade
ORDER BY ABS(soma_fluxos - variacao_caixa) DESC;
"""
with engine.connect() as conn:
    df_ident_dfc = pd.read_sql(text(query_identidade_dfc), conn)

n_diverge_dfc  = (df_ident_dfc['status_identidade'] == '❌ DIVERGE').sum()
n_sem_fluxo    = (df_ident_dfc['status_identidade'] == '⚠️ ZERADO').sum()
n_ok_ident_dfc = (df_ident_dfc['status_identidade'] == '✅ OK').sum()
n_cambial      = df_ident_dfc['status_identidade'].str.startswith('💱').sum()
n_desbal_dfc   = n_diverge_dfc

print("📊 RESULTADO — Equação Fundamental da DFC por Empresa/Data")
print(f"\n✅ OK:                          {n_ok_ident_dfc}")
print(f"💱 Efeito Cambial (6.04 impl.):  {n_cambial}")
print(f"⚠️  ZERADO:                       {n_sem_fluxo}")
print(f"❌ DIVERGE (erro de pipeline):   {n_diverge_dfc}")

if n_cambial > 0:
    print(f"\n💱 Caso(s) de efeito cambial (dado original, pipeline correto):")
    display(df_ident_dfc[df_ident_dfc['status_identidade'].str.startswith('💱')])

if n_diverge_dfc > 0:
    print(f"\n❌ FALHOU: {n_diverge_dfc} caso(s) com divergência real.")
    display(df_ident_dfc[df_ident_dfc['status_identidade'] == '❌ DIVERGE'])
else:
    print("\n✅ PASSOU: Identidade fundamental da DFC validada em todos os períodos.")

---
### 💱 Caso de Uso: ENGIE Brasil 2021 — O Efeito Cambial Silencioso

A ENGIE Brasil é subsidiária do grupo francês ENGIE. Em 2021, a variação do USD/BRL gerou um **ganho cambial de R$828 mil** sobre o saldo de caixa, embutido diretamente em `6.05` sem abrir `6.04` — prática permitida pelo CPC 03.

O teste classifica esse caso como `💱 Efeito Cambial` e **não reprova**, pois:
- `FLAG_RECONSTRUCAO = False` (dado original CVM)
- Divergência / 6.05 = 0,13% (abaixo do teto de 1%)

In [ ]:
CNPJ_ENGIE = '02.474.103/0001-19'
DT_ENGIE   = '2021-12-31'

query_engie = f"""
SELECT "CD_CONTA", "DS_CONTA",
       ROUND("VL_CONTA_TRATADO"::NUMERIC, 2) AS valor,
       "FLAG_RECONSTRUCAO"
FROM {TABELA_DFC}
WHERE "CNPJ_CIA" = '{CNPJ_ENGIE}'
  AND "DT_REFER"::DATE = '{DT_ENGIE}'
  AND "CD_CONTA" IN ('6.01','6.02','6.03','6.04','6.05','6.05.01','6.05.02')
ORDER BY "CD_CONTA";
"""

with engine.connect() as conn:
    df_engie = pd.read_sql(text(query_engie), conn)

if df_engie.empty:
    print("ℹ️  Dados da ENGIE Brasil 2021 não encontrados na base atual.")
else:
    print('ENGIE BRASIL ENERGIA S.A. --- 2021-12-31')
    print('=' * 66)
    for _, row in df_engie.iterrows():
        recon = ' [RECONSTRUIDO]' if row['FLAG_RECONSTRUCAO'] else ''
        print(f"   {row['CD_CONTA']:<10} {row['DS_CONTA']:<42} {row['valor']:>16,.0f}{recon}")

In [ ]:
query_status_math_dfc = f"""
SELECT
    "STATUS_MATH",
    COUNT(*)                                              AS qtd_linhas,
    COUNT(DISTINCT "CNPJ_CIA" || "DT_REFER"::TEXT)       AS qtd_empresa_data,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)   AS pct_total
FROM {TABELA_DFC}
GROUP BY "STATUS_MATH"
ORDER BY qtd_linhas DESC;
"""
with engine.connect() as conn:
    df_status_dfc = pd.read_sql(text(query_status_math_dfc), conn)

print("📊 RESULTADO — STATUS_MATH (DFC)")
display(df_status_dfc)

pct_desbal_status_dfc = df_status_dfc.loc[
    df_status_dfc['STATUS_MATH'].isin(['DESBALANCEADO', 'INCONSISTENTE']), 'pct_total'
].sum()

if pct_desbal_status_dfc > 5.0:
    print(f"\n❌ FALHOU: {pct_desbal_status_dfc:.1f}% das linhas DFC DESBALANCEADAS.")
else:
    print(f"\n✅ PASSOU: Apenas {pct_desbal_status_dfc:.1f}% desbalanceadas.")

In [ ]:
query_flags_dfc = f"""
SELECT
    "FLAG_RECONSTRUCAO", "FLAG_NORMALIZACAO",
    COUNT(*)                                            AS qtd_linhas,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_total
FROM {TABELA_DFC}
GROUP BY "FLAG_RECONSTRUCAO", "FLAG_NORMALIZACAO"
ORDER BY qtd_linhas DESC;
"""
with engine.connect() as conn:
    df_flags_dfc = pd.read_sql(text(query_flags_dfc), conn)

print("📊 RESULTADO — Flags de Qualidade (DFC)")
display(df_flags_dfc)

total_dfc_f = df_flags_dfc['qtd_linhas'].sum()
linhas_reconst_dfc = df_flags_dfc.loc[df_flags_dfc['FLAG_RECONSTRUCAO'] == True, 'qtd_linhas'].sum()
pct_reconst_dfc = (linhas_reconst_dfc / total_dfc_f * 100) if total_dfc_f > 0 else 0

if pct_reconst_dfc == 100:
    print("\n❌ ALERTA: 100% das linhas DFC reconstruídas!")
elif pct_reconst_dfc > 50:
    print(f"\n⚠️  ATENÇÃO: {pct_reconst_dfc:.1f}% das linhas DFC são reconstruídas.")
else:
    print(f"\n✅ PASSOU: {pct_reconst_dfc:.1f}% de linhas reconstruídas.")

In [ ]:
query_leaf_dfc = f"""
SELECT
    "IS_LEAF",
    COUNT(*)                                            AS qtd_linhas,
    COUNT(DISTINCT "CD_CONTA")                          AS contas_distintas,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_total
FROM {TABELA_DFC}
GROUP BY "IS_LEAF"
ORDER BY "IS_LEAF";
"""
with engine.connect() as conn:
    df_leaf_dfc = pd.read_sql(text(query_leaf_dfc), conn)

print("📊 RESULTADO — IS_LEAF (DFC)")
display(df_leaf_dfc)

valores_leaf_dfc = df_leaf_dfc['IS_LEAF'].astype(str).str.lower().tolist()
tem_nao_leaf_dfc = any(v in ['false', '0'] for v in valores_leaf_dfc)
tem_leaf_dfc     = any(v in ['true', '1'] for v in valores_leaf_dfc)

if not tem_nao_leaf_dfc:
    print("\n❌ FALHOU: Todas as contas DFC são IS_LEAF = True. Nós pai estão faltando!")
elif not tem_leaf_dfc:
    print("\n❌ FALHOU: Nenhuma conta IS_LEAF = True na DFC.")
else:
    print("\n✅ PASSOU: Árvore DFC com contas analíticas (folhas) e sintéticas (pais).")

In [ ]:
query_duplicatas_dfc = f"""
SELECT "CNPJ_CIA", "DENOM_CIA", "DT_REFER", "CD_CONTA", "DS_CONTA", COUNT(*) AS ocorrencias
FROM {TABELA_DFC}
GROUP BY "CNPJ_CIA", "DENOM_CIA", "DT_REFER", "CD_CONTA", "DS_CONTA"
HAVING COUNT(*) > 1
ORDER BY ocorrencias DESC LIMIT 20;
"""
with engine.connect() as conn:
    df_dupl_dfc = pd.read_sql(text(query_duplicatas_dfc), conn)

print("📊 RESULTADO — Duplicatas DFC (esperado: zero linhas)")
if df_dupl_dfc.empty:
    print("✅ PASSOU: Nenhuma duplicata na DFC.")
else:
    display(df_dupl_dfc)
    print(f"\n❌ FALHOU: {len(df_dupl_dfc)} grupos de duplicatas na DFC.")

---
## 💰 BLOCO 7 — Cross-check DFC ↔ BP: `6.05.02` vs `1.01.01 + 1.01.02`

Três fenômenos distintos explicam divergências estruturais:

| Norma | Descrição | Impacto |
|---|---|---|
| **CPC 03 / IAS 7** | Aplicações ≤90 dias (CDBs, LFTs) são caixa na DFC mas ficam em `1.01.02` no BP | DFC > `1.01.01` mas ≤ `1.01.01 + 1.01.02` |
| **CPC 31 / IFRS 5** | Caixa de grupos a venda migra para `1.01.08.01/02` no BP mas permanece em `6.05.02` na DFC | DFC > `1.01.01 + 1.01.02` |
| **Caixa Restrito** | BP `1.01.01` inclui caixa restrito excluído da DFC | DFC < `1.01.01` |

> **Regra de ouro:** Use `6.05.02` para Dívida Líquida — sempre.

In [ ]:
query_cross_dfc_bp = f"""
WITH dfc_saldo AS (
    SELECT "CNPJ_CIA", "DENOM_CIA", "DT_REFER",
           SUM("VL_CONTA_TRATADO")      AS saldo_dfc,
           BOOL_OR("FLAG_RECONSTRUCAO") AS reconstruido
    FROM {TABELA_DFC}
    WHERE "CD_CONTA" = '6.05.02'
    GROUP BY "CNPJ_CIA", "DENOM_CIA", "DT_REFER"
),
bp_caixa AS (
    SELECT "CNPJ_CIA", "DT_REFER",
        SUM(CASE WHEN "CD_CONTA"='1.01.01' THEN "VL_CONTA_TRATADO" ELSE 0 END) AS c1,
        SUM(CASE WHEN "CD_CONTA"='1.01.02' THEN "VL_CONTA_TRATADO" ELSE 0 END) AS c2
    FROM {TABELA_BP}
    WHERE "CD_CONTA" IN ('1.01.01','1.01.02')
    GROUP BY "CNPJ_CIA", "DT_REFER"
)
SELECT
    d."CNPJ_CIA", d."DENOM_CIA", d."DT_REFER",
    ROUND(d.saldo_dfc::NUMERIC, 2)                     AS dfc_6_05_02,
    ROUND(b.c1::NUMERIC, 2)                            AS bp_1_01_01,
    ROUND(COALESCE(b.c2, 0)::NUMERIC, 2)               AS bp_1_01_02,
    ROUND((b.c1 + COALESCE(b.c2, 0))::NUMERIC, 2)     AS bp_teto_cpc03,
    d.reconstruido                                     AS dfc_reconstruido,
    CASE
        WHEN b.c1 IS NULL                                                         THEN '⚠️ SEM BP'
        WHEN ABS(d.saldo_dfc - b.c1) <= 1000                                     THEN '✅ Idêntico a 1.01.01'
        WHEN d.saldo_dfc > b.c1 AND d.saldo_dfc <= b.c1 + COALESCE(b.c2,0)+1000 THEN '✅ CPC 03 válido (CDB/LFT em 1.01.02)'
        WHEN d.saldo_dfc < b.c1 - 1000                                           THEN '✅ Caixa restrito no BP'
        WHEN d.saldo_dfc > b.c1 + COALESCE(b.c2,0)+1000 AND d.reconstruido=FALSE THEN '🏛 CPC 31/IFRS 5 (grupos a venda)'
        WHEN d.saldo_dfc > b.c1 + COALESCE(b.c2,0)+1000 AND d.reconstruido=TRUE  THEN '❌ DFC supera teto — dado RECONSTRUÍDO'
        ELSE '⚠️ Verificar'
    END AS diagnostico
FROM dfc_saldo d
LEFT JOIN bp_caixa b ON d."CNPJ_CIA"=b."CNPJ_CIA" AND d."DT_REFER"=b."DT_REFER"
ORDER BY
    CASE WHEN d.reconstruido AND d.saldo_dfc > b.c1 + COALESCE(b.c2,0)+1000 THEN 0
         WHEN d.saldo_dfc > b.c1 + COALESCE(b.c2,0)+1000 THEN 1 ELSE 2 END,
    ABS(d.saldo_dfc - COALESCE(b.c1,0)) DESC;
"""
with engine.connect() as conn:
    df_cross_bp = pd.read_sql(text(query_cross_dfc_bp), conn)

print("📊 DIAGNÓSTICO — DFC 6.05.02 ↔ BP (intervalo CPC 03 + CPC 31)")
print(f"   Total de pares analisados: {len(df_cross_bp)}\n")

dist = df_cross_bp['diagnostico'].value_counts()
for status, qtd in dist.items():
    pct = qtd / len(df_cross_bp) * 100
    print(f"   {status:<58} {qtd:>5} ({pct:.1f}%)")

df_erro = df_cross_bp[df_cross_bp['diagnostico'].str.startswith('❌ DFC supera')]
n_erro = len(df_erro)
ok7_dfc = (n_erro == 0)

df_cpc31 = df_cross_bp[df_cross_bp['diagnostico'].str.startswith('🏛 CPC 31')]
if not df_cpc31.empty:
    print(f"\n   🏛 {len(df_cpc31)} caso(s) CPC 31/IFRS 5 (caixa em grupos a venda — dado original CVM):")
    display(df_cpc31[["CNPJ_CIA","DENOM_CIA","DT_REFER","dfc_6_05_02","bp_1_01_01","bp_teto_cpc03","dfc_reconstruido"]].head(10))

if n_erro > 0:
    print(f"\n   ❌ {n_erro} ERRO(S) DE PIPELINE — dado reconstruído acima do teto:")
    display(df_erro)
else:
    print("\n   ✅ Nenhum erro de pipeline (nenhum dado reconstruído acima do teto CPC 03).")

status7 = "✅ PASSOU" if ok7_dfc else "❌ FALHOU"
print(f"\n{status7} — Bloco 7: erros de pipeline: {n_erro}")

In [ ]:
print("=" * 68)
print("  RELATÓRIO DE SANIDADE — layer_02_silver.n1_dfc")
print("=" * 68)

resultados_dfc = [
    ("1. Visão Geral (volume e nulos)", (total_dfc > 0) and (nulls_dfc == 0)),
    ("2. Equação Fundamental (6.01+6.02+6.03+6.04 = 6.05)", n_desbal_dfc == 0),
    ("3. STATUS_MATH (≤5% desbalanceado)", pct_desbal_status_dfc <= 5.0),
    ("4. Flags de Qualidade (não 100% reconstruído)", pct_reconst_dfc < 100),
    ("5. IS_LEAF (árvore com pais e folhas)", tem_leaf_dfc and tem_nao_leaf_dfc),
    ("6. Duplicatas (zero registros)", df_dupl_dfc.empty),
    ("7. DFC ↔ BP: sem erro de pipeline CPC 31", ok7_dfc),
]

aprovados_dfc = 0
for nome, passou in resultados_dfc:
    icone = "✅ PASSOU" if passou else "❌ FALHOU"
    print(f"  {icone}  |  {nome}")
    if passou:
        aprovados_dfc += 1

if 'df_cross_bp' in dir():
    print()
    n_cpc03  = len(df_cross_bp[df_cross_bp["diagnostico"].str.startswith("✅ CPC 03")])
    n_cpc31  = len(df_cross_bp[df_cross_bp["diagnostico"].str.startswith("🏛 CPC 31")])
    n_restri = len(df_cross_bp[df_cross_bp["diagnostico"].str.startswith("✅ Caixa restrito")])
    if n_cpc03:  print(f"  ℹ️ INFORMACIONAL  |  CPC 03 — {n_cpc03} caso(s) com equiv. de caixa em 1.01.02")
    if n_cpc31:  print(f"  ℹ️ INFORMACIONAL  |  CPC 31/IFRS 5 — {n_cpc31} caso(s) com caixa em grupos a venda")
    if n_restri: print(f"  ℹ️ INFORMACIONAL  |  Caixa Restrito — {n_restri} caso(s)")

print("=" * 68)
print(f"  RESULTADO FINAL: {aprovados_dfc}/{len(resultados_dfc)} testes obrigatórios aprovados")
if aprovados_dfc == len(resultados_dfc):
    print("  🎉 DFC APROVADA — dados prontos para análise financeira!")
else:
    print("  🚨 DFC REPROVADA — revisar o pipeline Silver antes de usar.")
print("=" * 68)